# tinyforge d6 on one A100

Trains the nanochat-d6 recipe (depth 6, width 384, 3 heads, vocab 32768, context 2048, 262144 tokens/step, 1770 steps) on ClimbMix. Val metric is bits per byte.

The repo is private. Put a GitHub personal access token with `repo` scope in the next cell, or skip it if you have already authenticated. Nothing is copied to Drive except the checkpoint, the tokenizer, and the log. Parquet shards and the token file stay on the Colab disk.

Read `docs/DESIGN.md` before a long run. The CUDA backward is behind the CPU oracle, and the attention kernel is one block per query, so the first A100 pass may take several hours rather than nanochat's ~50 minutes. The smoke test prints tokens/sec. If that is far below 1e5, stop and look at attention before spending the 1770 steps.


In [ ]:
# @title GitHub token (private repo)
import os
GITHUB_TOKEN = ""  # @param {type:"string"}
REPO = "michaelp91-dev/tinyforge"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN
os.environ["REPO"] = REPO
os.environ["BRANCH"] = BRANCH
print("repo", REPO, "branch", BRANCH, "token set" if GITHUB_TOKEN else "NO TOKEN")


In [ ]:
%%bash
set -euo pipefail
cd /content
if [ -n "${GITHUB_TOKEN}" ]; then
  git clone --branch "${BRANCH}" "https://${GITHUB_TOKEN}@github.com/${REPO}.git" tinyforge
else
  echo "No token. Trying an unauthenticated clone (works only if the repo is public)."
  git clone --branch "${BRANCH}" "https://github.com/${REPO}.git" tinyforge
fi
cd tinyforge
make -j$(nproc)
make cuda -j$(nproc)
ls -lh build/tf_train_cuda build/tf_tok_train


## Data

Train shards `00000`–`00040`, val shard `06542`, from `karpathy/climbmix-400b-shuffle`. About 3.9GB. Not copied to Drive.


In [ ]:
%%bash
set -euo pipefail
cd /content/tinyforge
mkdir -p data/parquet
BASE=https://huggingface.co/datasets/karpathy/climbmix-400b-shuffle/resolve/main
for i in $(seq 0 40); do
  f=$(printf "shard_%05d.parquet" "$i")
  wget -q -c -O "data/parquet/$f" "$BASE/$f" &
  # a few at a time
  if (( i % 4 == 3 )); then wait; fi
done
wait
wget -q -c -O data/parquet/shard_06542.parquet "$BASE/shard_06542.parquet"
ls data/parquet | wc -l


## Tokenizer

Trains a 32,768 BPE on the first 200MB of text from shard 0. That is a sample, not nanochat's full tokenizer corpus. bpb is still the comparable metric.


In [ ]:
%%bash
set -euo pipefail
cd /content/tinyforge
./build/tf_tok_train --vocab 32768 --max-bytes 200000000 --threads 8   --out tokenizer.bin data/parquet/shard_00000.parquet
ls -lh tokenizer.bin


In [ ]:
%%bash
set -euo pipefail
cd /content/tinyforge
# train shards then the val shard, as two token files
./build/tf_tok_encode --tokenizer tokenizer.bin --threads 8 --out data/train.bin   data/parquet/shard_000{00,01,02,03,04,05,06,07,08,09}.parquet   data/parquet/shard_000{10,11,12,13,14,15,16,17,18,19}.parquet   data/parquet/shard_000{20,21,22,23,24,25,26,27,28,29}.parquet   data/parquet/shard_000{30,31,32,33,34,35,36,37,38,39}.parquet   data/parquet/shard_00040.parquet
./build/tf_tok_encode --tokenizer tokenizer.bin --threads 8 --out data/val.bin   data/parquet/shard_06542.parquet
ls -lh data/train.bin data/val.bin


## Smoke test

Twenty steps of a tiny model on the real token file. Confirm the loss is finite and read tokens/sec before the full run.


In [ ]:
%%bash
set -euo pipefail
cd /content/tinyforge
./build/tf_train_cuda --backend cuda   --tokenizer tokenizer.bin --data data/train.bin --val data/val.bin   --depth 2 --dim 128 --heads 1 --seq 256   --device-batch 4 --total-batch 1024 --steps 20   --eval-every 20 --eval-tokens 4096   --out runs/smoke
tail -n 30 runs/smoke/train.log


## d6

1,770 steps, 262,144 tokens/step, microbatch 8. Writes `runs/d6/ckpt.bin`, `ckpt.bin.optim`, and `train.log`.


In [ ]:
%%bash
set -euo pipefail
cd /content/tinyforge
./build/tf_train_cuda --backend cuda   --tokenizer tokenizer.bin --data data/train.bin --val data/val.bin   --depth 6 --head-dim 128 --seq 2048 --ratio 20   --device-batch 8 --total-batch 262144   --eval-every 250 --eval-tokens 2097152   --save-every 250   --out runs/d6
echo "---- log tail ----"
tail -n 40 runs/d6/train.log


In [ ]:
%%bash
set -euo pipefail
cd /content/tinyforge
./build/tf_sample_cuda --backend cuda   --ckpt runs/d6/ckpt.bin --tokenizer tokenizer.bin   --prompt "The capital of France is" --tokens 64 --temp 0.8 --topk 40 --seed 1


## Copy lean outputs to Drive

Weights are about 300MB, the optimizer state about 550MB more. If Drive is tight, delete `ckpt.bin.optim` from the copy. Do not copy `data/`.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
%%bash
set -euo pipefail
DEST=/content/drive/MyDrive/tinyforge
mkdir -p "$DEST"
cd /content/tinyforge
cp -n tokenizer.bin "$DEST/" || cp tokenizer.bin "$DEST/tokenizer.bin"
cp runs/d6/train.log "$DEST/train.log"
cp runs/d6/ckpt.bin "$DEST/ckpt.bin"
# optimizer state is optional; comment the next line if Drive is full
cp runs/d6/ckpt.bin.optim "$DEST/ckpt.bin.optim" || echo "skipped optim state"
ls -lh "$DEST"
